# Friction Score

Notebook hướng dẫn tính Friction Score cho các task IT và phân tích theo nhóm.

In [1]:
%pip install -U kaleido

import sys
print('Kernel python executable:', sys.executable)

Note: you may need to restart the kernel to use updated packages.
Kernel python executable: c:\Users\THINHKPAD\anaconda3\python.exe


### 1. Imports và cấu hình môi trường

In [ ]:
import sys
from pathlib import Path
cwd = Path.cwd()
if (cwd / 'src').exists():
    ROOT = cwd
elif (cwd.parent / 'src').exists():
    ROOT = cwd.parent
elif (cwd.parent.parent / 'src').exists():
    ROOT = cwd.parent.parent
else:
    ROOT = cwd
    print('WARNING: Không tìm thấy thư mục src trên các mức dự đoán. Sử dụng cwd:', ROOT)

sys.path.insert(0, str(ROOT))

import importlib.util
import pandas as pd
import plotly.express as px
from src.friction_score import build_friction_outputs, build_friction_table

print('kaleido installed:', importlib.util.find_spec('kaleido') is not None)

pd.set_option('display.max_columns', 200)

kaleido installed: True


### 2. Chạy pipeline và kết quả

In [3]:
INPUT_PATH = ROOT / 'data' / 'processed' / 'it_master.csv'
WORKER_PATH = ROOT / 'data' / 'processed' / 'it_worker_level.csv'

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Chưa có dữ liệu đầu vào: {INPUT_PATH}. Hãy chạy src.data_processing trước.')

master = pd.read_csv(INPUT_PATH)
friction_table = build_friction_table(master)

# Try to read worker-level table if exists for by-group analysis, otherwise let pipeline produce it
if WORKER_PATH.exists():
    worker_level = pd.read_csv(WORKER_PATH)
else:
    worker_level = None

# Also compute by-group using function if worker_level provided
if worker_level is not None:
    from src.friction_score import build_friction_by_group
    friction_by_group = build_friction_by_group(friction_table, worker_level)
else:
    # fallback: run full pipeline (reads files internally)
    tables = build_friction_outputs()
    friction_by_group = tables.get('friction_score_by_group')

print(f'Loaded master: {INPUT_PATH} -> {master.shape[0]} rows × {master.shape[1]} cols')
print(f'Computed friction_table: {friction_table.shape[0]} rows × {friction_table.shape[1]} cols')

friction_table.head(10)


[INFO] friction_score: loai 55/186 task IT vi thieu Expert hoac Worker rating (khong du dieu kien tinh Friction Score).
[INFO] friction_score: nguong canh bao do = 47.64 diem (top 25%), 33/131 task duoc gan co Do.


Loaded master: c:\Users\THINHKPAD\Documents\AI_Agent_Project\data\processed\it_master.csv -> 186 rows × 49 cols
Computed friction_table: 131 rows × 14 cols


,Task ID,Occupation (O*NET-SOC Title),Task,Expert_Automation Capacity Rating,Worker_Automation Desire Rating,Capacity_Desire_Gap_Raw,Worker_Job Security Rating,Worker_Enjoyment Rating,HumanAgency_Resistance_Raw,Friction Score,Canh_Bao,Lý do chính,Expert_N_Raters,Worker_N_Respondents
0,14647,Software Quality Assurance Analysts and Testers,Review software documentation to ensure techni...,4.500000,2.333333,2.166667,1.666667,2.666667,0.444444,66.54,Đỏ,Lo ngại mất việc (Job Security),2.0,3.0
1,14671,Computer Systems Engineers/Architects,Evaluate current or emerging technologies to c...,3.000000,2.600000,0.400000,2.200000,3.600000,0.733333,63.97,Đỏ,Muốn giữ vai trò con người (Control/Empathy/Et...,3.0,5.0
2,5317,Information Security Analysts,Modify computer security files to incorporate ...,3.500000,2.625000,0.875000,1.500000,3.875000,0.291667,63.00,Đỏ,Gắn bó/yêu thích task (Enjoyment),2.0,8.0
3,14681,Computer Systems Engineers/Architects,"Define and analyze objectives, scope, issues, ...",2.666667,2.000000,0.666667,1.666667,4.000000,0.333333,62.01,Đỏ,Gắn bó/yêu thích task (Enjoyment),3.0,3.0
4,1285,Computer User Support Specialists,Oversee the daily performance of computer syst...,5.000000,2.750000,2.250000,2.375000,3.250000,0.333333,61.54,Đỏ,Chênh lệch AI làm được vs người lao động muốn,2.0,8.0
5,18991,Computer Network Support Specialists,Configure security settings or access permissi...,4.666667,2.333333,2.333333,2.166667,2.666667,0.333333,58.99,Đỏ,Chênh lệch AI làm được vs người lao động muốn,3.0,6.0
6,14644,Software Quality Assurance Analysts and Testers,Create or maintain databases of known test def...,5.000000,3.250000,1.750000,1.500000,2.750000,0.250000,58.76,Đỏ,Lo ngại mất việc (Job Security),2.0,4.0
7,1313,Database Administrators,Write and code logical and physical database d...,4.500000,1.666667,2.833333,2.666667,2.666667,0.333333,58.17,Đỏ,Chênh lệch AI làm được vs người lao động muốn,2.0,3.0
8,14641,Software Quality Assurance Analysts and Testers,"Document software defects, using a bug trackin...",4.000000,2.666667,1.333333,2.000000,3.333333,0.333333,58.16,Đỏ,Gắn bó/yêu thích task (Enjoyment),3.0,6.0
9,14650,Software Quality Assurance Analysts and Testers,Update automated test scripts to ensure currency.,4.500000,3.666667,0.833333,2.000000,3.333333,0.444444,57.53,Đỏ,Gắn bó/yêu thích task (Enjoyment),2.0,3.0


### 3. Các Task có Friction cao nhất (Top 10)

In [ ]:
top10 = friction_table.head(10).copy()

# Rút gọn tên task còn khoảng 40 ký tự
top10["Task_Short"] = top10["Task"].apply(
    lambda x: x[:40] + "..." if len(x) > 40 else x
)

fig = px.bar(
    top10,
    x="Friction Score",
    y="Task_Short",
    orientation="h",
    color="Lý do chính",
    text="Friction Score",
    hover_data=["Task"],
    title="Top 10 Task IT có Friction Score cao nhất"
)

fig.update_traces( texttemplate="%{text:.1f}", textposition="outside", cliponaxis=False)

fig.update_layout(
    template="plotly_white", width=1200, height=650, title=dict(x=0.5, font=dict(size=24)),

    xaxis=dict(
        title="Friction Score",
        range=[0,100],
        tickfont=dict(size=12)
    ),

    yaxis=dict( title="", categoryorder="total ascending", tickfont=dict(size=11), automargin=True),

    legend=dict(orientation="h", y=1.08, x=0.5, xanchor="center"),

    margin=dict(l=280, r=60, t=120, b=60)
)

fig.show()


Không lưu được ảnh tĩnh bằng kaleido: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

Đã lưu thay thế định dạng HTML: c:\Users\THINHKPAD\Documents\AI_Agent_Project\docs\screenshots\Top 10 Task Friction Score.html


c:\Users\THINHKPAD\anaconda3\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




### Phân bố Friction Score

Biểu đồ histogram giúp kiểm tra xem điểm phân bố như thế nào (tập trung, lệch phải/trái, có nhiều giá trị 0/100 không).

In [27]:
hist = friction_table.copy()

fig_hist = px.histogram(
    hist, 
    x='Friction Score', 
    nbins=20, 
    color='Canh_Bao',
    title='Phân bố Friction Score trên các task IT',
    labels={'Friction Score': 'Điểm Friction Score', 'count': 'Số lượng task'}
)


nguong_do = hist['Friction Score'].quantile(0.75)
fig_hist.add_vline(
    x=nguong_do, 
    line_dash='dash', 
    line_color='red',
    annotation_text=f'Ngưỡng Q75 ({nguong_do:.2f})'
)

fig_hist.update_layout(height=400)
fig_hist.show()

### Chênh lệch AI vs Worker so với Friction Score

Scatter plot giữa `Capacity_Desire_Gap_Raw` và `Friction Score` để kiểm tra mối quan hệ trực quan giữa khoảng cách năng lực/khát vọng và lực cản.

In [ ]:
# Scatter: Capacity_Desire_Gap_Raw vs Friction Score
if 'Capacity_Desire_Gap_Raw' in friction_table.columns:
    scatter_df = friction_table.dropna(subset=['Capacity_Desire_Gap_Raw'])
    fig_scatter = px.scatter(
        scatter_df,
        x='Capacity_Desire_Gap_Raw',
        y='Friction Score',
        color='Canh_Bao',
        hover_data=['Task', 'Lý do chính'],
        trendline='ols',
        title='Capacity-Desire Gap vs Friction Score'
    )
    fig_scatter.update_layout(height=450)
    fig_scatter
else:
    print('Không tìm thấy cột Capacity_Desire_Gap_Raw trong bảng friction_table.')


Không lưu được ảnh (thiếu engine hoặc lỗi xuất ảnh): 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

Tiếp tục hiển thị trong notebook.


### Top 'Lý do chính' đóng góp vào Friction

Đếm tần suất các `Lý do chính` để xem thành phần nào thường xuyên là nguyên nhân chính dẫn tới friction cao.

In [ ]:
reasons = friction_table['Lý do chính'].value_counts().reset_index()
reasons.columns = ['Lý do chính', 'Count']
fig_reasons = px.bar(reasons, x='Count', y='Lý do chính', orientation='h', text='Count', title="Top 'Lý do chính' cho Friction")
fig_reasons.update_layout(height=400)
fig_reasons


Không lưu được ảnh (thiếu engine hoặc lỗi xuất ảnh): 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

Tiếp tục hiển thị trong notebook.


### 4. Phân tích theo nhóm (friction_score_by_group)

In [30]:
friction_by_group.head(20)

,Nhom_Phan_Tich,Gia_Tri,Avg_Friction_Score,Pct_Canh_Bao_Do,N_Worker_Responses
0,AI Job Importance Attitude,Somewhat disagree,42.499,27.300,227
1,AI Job Importance Attitude,Neither agree nor disagree,42.047,24.000,167
2,AI Job Importance Attitude,Strongly disagree,41.630,26.700,146
3,AI Job Importance Attitude,Somewhat agree,40.907,20.100,313
4,AI Job Importance Attitude,Strongly agree,40.865,22.800,149
5,AI Suffering Attitude,Strongly disagree,41.832,24.300,239
6,AI Suffering Attitude,Somewhat disagree,41.637,23.800,341
7,AI Suffering Attitude,Neither agree nor disagree,41.568,24.100,166
8,AI Suffering Attitude,Somewhat agree,41.476,25.100,199
9,AI Suffering Attitude,Strongly agree,40.170,15.800,57
